In [1]:
import numpy as np
import time
import random
import jsonlines
import weaviate
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# -----------------------------------------------
# Set random seed for reproducibility
# -----------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

c:\Users\zzhen\projects\dsa4213-ay2526-group41\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load queries and qrels from JSONL files
QUERIES_QRELS_PATH = "../data/processed/formatted_train_data.jsonl"

def load_queries_and_qrels(queries_qrels_path):
    queries, qrels = {}, {}

    with jsonlines.open(queries_qrels_path) as reader:
        for obj in reader:
            queries[obj["qid"]] = obj["query"]
            qrels[obj["qid"]] = set(obj["qrels"])

    return queries, qrels

queries, qrels = load_queries_and_qrels(QUERIES_QRELS_PATH)
print(f"Loaded {len(queries)} queries and {len(qrels)} qrels.")

Loaded 33346 queries and 33346 qrels.


In [3]:
# Connect to Weaviate instance
HOST = "127.0.0.1"
PORT = 8080
GRPC_PORT = 50051
COLLECTION_NAME = "MS_MARCO"

client = weaviate.connect_to_local(
    host=HOST,
    port=PORT,
    grpc_port=GRPC_PORT,
)
print(f"Connected to Weaviate at {HOST}:{PORT} with gRPC port {GRPC_PORT}.")

# Use collection
collection = client.collections.use(COLLECTION_NAME)
print(f"Using collection: {COLLECTION_NAME}")

Connected to Weaviate at 127.0.0.1:8080 with gRPC port 50051.
Using collection: MS_MARCO


In [4]:
# Load pre-trained embedding model
model = SentenceTransformer(
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
)
print("Loaded pre-trained embedding model.")

Loaded pre-trained embedding model.


In [5]:
all_latencies = []
all_accuracies = []

# Do a BM25 search
for qid, query_text in tqdm(queries.items(), desc="BM25 Search"):
    # Start timer
    start_time = time.time()

    # Perform BM25 search
    response = collection.query.bm25(
        query=query_text,
        limit=5,
        return_properties=["pid"]
    )

    # End timer
    end_time = time.time()
    latency = (end_time - start_time) * 1000  # in milliseconds
    all_latencies.append(latency)

    # Compute accuracy
    retrieved_pids = [
        obj.properties["pid"] for obj in response.objects
    ]
    relevant_pids = qrels[qid]
    correct_retrievals = len(set(retrieved_pids) & relevant_pids)
    accuracy = correct_retrievals / len(relevant_pids) if relevant_pids else 0
    all_accuracies.append(accuracy)

print("Completed BM25 search for all queries.")

BM25 Search: 100%|██████████| 33346/33346 [00:59<00:00, 563.81it/s]

Completed BM25 search for all queries.


In [6]:
# Compute average latency and accuracy
total_latencies = np.sum(all_latencies)
first_latency = all_latencies[0] # Use this a refernce because weaviate has cache for multiple queries that are the same
avg_latency = np.mean(all_latencies)
avg_accuracy = np.mean(all_accuracies)
print(f"Total Latency: {total_latencies:.4f} ms")
print(f"First Latency: {first_latency:.4f} ms")
print(f"Average Latency: {avg_latency:.4f} ms")
print(f"Average Accuracy: {avg_accuracy:.4f}")

Total Latency: 58702.3401 ms
First Latency: 3.0332 ms
Average Latency: 1.7604 ms
Average Accuracy: 0.6821


In [7]:
all_latencies = []
all_accuracies = []

# Do a dense vector search
for qid, query_text in tqdm(queries.items(), desc="Dense Vector Search"):
    # Start timer
    start_time = time.time()

    # Compute query embedding
    query_embedding = model.encode(query_text).tolist()

    # Perform vector search, weaviate use cosine similarity by default
    response = collection.query.near_vector(
        near_vector=query_embedding,
        limit=5,
        return_properties=["pid"]
    )

    # End timer
    end_time = time.time()
    latency = (end_time - start_time) * 1000  # in milliseconds
    all_latencies.append(latency)

    # Compute accuracy
    retrieved_pids = [
        obj.properties["pid"] for obj in response.objects
    ]
    relevant_pids = qrels[qid]
    correct_retrievals = len(set(retrieved_pids) & relevant_pids)
    accuracy = correct_retrievals / len(relevant_pids) if relevant_pids else 0
    all_accuracies.append(accuracy)

print("Completed dense vector search for all queries.")

Dense Vector Search: 100%|██████████| 33346/33346 [04:05<00:00, 135.98it/s]

Completed dense vector search for all queries.


In [8]:
# Compute average latency and accuracy
total_latencies = np.sum(all_latencies)
first_latency = all_latencies[0] # Use this a refernce because weaviate has cache for multiple queries that are the same
avg_latency = np.mean(all_latencies)
avg_accuracy = np.mean(all_accuracies)
print(f"Total Latency: {total_latencies:.4f} ms")
print(f"First Latency: {first_latency:.4f} ms")
print(f"Average Latency: {avg_latency:.4f} ms")
print(f"Average Accuracy: {avg_accuracy:.4f}")

Total Latency: 243835.0132 ms
First Latency: 187.9113 ms
Average Latency: 7.3123 ms
Average Accuracy: 0.9026


In [9]:
all_latencies = []
all_accuracies = []

# Do a hybrid search
for qid, query_text in tqdm(queries.items(), desc="Hybrid Search"):
    # Start timer
    start_time = time.time()

    # Compute query embedding
    query_embedding = model.encode(query_text).tolist()

    # Perform hybrid search, weaviate use cosine similarity by default
    response = collection.query.hybrid(
        query=query_text,
        vector=query_embedding,
        alpha=0.5,
        limit=5,
        return_properties=["pid"]
    )

    # End timer
    end_time = time.time()
    latency = (end_time - start_time) * 1000  # in milliseconds
    all_latencies.append(latency)

    # Compute accuracy
    retrieved_pids = [
        obj.properties["pid"] for obj in response.objects
    ]
    relevant_pids = qrels[qid]
    correct_retrievals = len(set(retrieved_pids) & relevant_pids)
    accuracy = correct_retrievals / len(relevant_pids) if relevant_pids else 0
    all_accuracies.append(accuracy)

print("Completed hybrid search for all queries.")

Hybrid Search: 100%|██████████| 33346/33346 [07:24<00:00, 75.10it/s] 

Completed hybrid search for all queries.


In [10]:
# Compute average latency and accuracy
total_latencies = np.sum(all_latencies)
first_latency = all_latencies[0] # Use this a refernce because weaviate has cache for multiple queries that are the same
avg_latency = np.mean(all_latencies)
avg_accuracy = np.mean(all_accuracies)
print(f"Total Latency: {total_latencies:.4f} ms")
print(f"First Latency: {first_latency:.4f} ms")
print(f"Average Latency: {avg_latency:.4f} ms")
print(f"Average Accuracy: {avg_accuracy:.4f}")

Total Latency: 441711.2327 ms
First Latency: 16.7263 ms
Average Latency: 13.2463 ms
Average Accuracy: 0.9005
